# File này để bơm dữ liệu vào database đánh giá model

## Pipeline bao gồm:
- Tạo kết nối với database evaluation
- Khai báo hàm tạo dữ liệu
- Thực thi hàm tạo dữ liệu

In [1]:
import os
import sqlite3
import random
from faker import Faker

### Tạo kết nối với database evaluation

In [2]:
data_path = os.path.join(os.path.dirname(os.path.dirname(os.path.abspath("__file__"))), "data")
conn = sqlite3.connect(os.path.join(data_path, "EA_eval.db"))
cursor = conn.cursor()

### Khai báo hàm tạo dữ liệu

#### Hàm tạo dữ liệu cho bảng Department

In [3]:
# Hàm tạo department
def feed_department_data(n_rows, cursor):
    fake = Faker('en_US')

    cursor.execute("SELECT MAX(department_id) FROM Department")
    result = cursor.fetchall()
    current_max_id = result[0][0] if result[0][0] is not None else 0
    print(f"Current max id: {current_max_id}")

    dept_prefixes = ["Department of", "Faculty of", "School of"]
    dept_subjects = [
        "Computer Science", "Economics", "Business Administration", 
        "Mathematics", "Physics", "Chemistry", "Biology", "Environmental Science",
        "Linguistics", "Law", "Philosophy", "Psychology", "Medicine", 
        "Mechanical Engineering", "Electrical Engineering", "Civil Engineering", 
        "Architecture", "Education", "Fine Arts", "History", "Sociology"
    ]

    fake_data = []

    for i in range(1, n_rows+1):
        department_id = current_max_id + i
        prefix = random.choice(dept_prefixes)
        subject = random.choice(dept_subjects)
        department_name = f"{prefix} {subject} - {fake.unique.random_number(digits=3)}"
        building = f"{fake.last_name()} Hall"
        foundation_year = random.randint(1850, 2023)
        fake_data.append((department_id, department_name, building, foundation_year))

    cursor.executemany('''
            INSERT INTO Department (department_id, department_name, building, foundation_year)
            VALUES (?, ?, ?, ?)
        ''', fake_data)
    
    print(f"Thêm {n_rows} dòng dữ liệu vào bảng Department thành công !")

#### Hàm tạo dữ liệu cho bảng Professor

In [4]:
# Hàm tạo Professor
def feed_professor_data(n_rows, cursor):
    fake = Faker('en_US')
    cursor.execute("SELECT department_id FROM Department")
    department_records = cursor.fetchall()

    if not department_records:
        print("Lỗi: Bảng Department đang trống! Bạn cần chạy hàm feed_department_data trước để có dữ liệu cấp cho Khóa ngoại.")
        conn.close()
        return

    valid_department_ids = [record[0] for record in department_records]

    cursor.execute("SELECT MAX(professor_id) FROM Professor")
    result = cursor.fetchone()
    current_max_id = result[0] if result[0] is not None else 0

    fake_data = []

    for i in range(1, n_rows+1):
        professor_id = current_max_id + i
        first_name = fake.first_name()
        last_name = fake.last_name()

        unique_suffix = fake.unique.random_int(min=1, max=99999)

        email = f"{first_name.lower()}.{last_name.lower()}{unique_suffix}@university.edu"
        salary = round(random.uniform(60000.00, 180000.00), 2)
        department_id = random.choice(valid_department_ids)
        fake_data.append((professor_id, first_name, last_name, email, salary, department_id))

    cursor.executemany('''
        INSERT INTO Professor (professor_id, first_name, last_name, email, salary, department_id)
        VALUES (?, ?, ?, ?, ?, ?)
    ''', fake_data)

    print(f"Thêm {n_rows} dòng dữ liệu vào bảng Professor thành công !")

#### Hàm tạo dữ liệu cho bảng Student

In [5]:
# Hàm tạo Student
def feed_student_data(n, cursor):
    fake = Faker('en_US')

    # 1. Lấy danh sách department_id để làm Khóa ngoại
    cursor.execute("SELECT department_id FROM Department")
    dept_records = cursor.fetchall()
    valid_dept_ids = [r[0] for r in dept_records] if dept_records else []

    # 2. Lấy student_id lớn nhất hiện tại
    cursor.execute("SELECT MAX(student_id) FROM Student")
    result = cursor.fetchone()
    current_max_id = result[0] if result[0] is not None else 0

    fake_data = []
    
    for i in range(1, n + 1):
        student_id = current_max_id + i
        first_name = fake.first_name()
        last_name = fake.last_name()
        
        # Email chuẩn sinh viên
        unique_suffix = fake.unique.random_int(1, 999999)
        email = f"{first_name.lower()}.{last_name.lower()}{unique_suffix}@student.university.edu"
        
        # --- DỮ LIỆU CHUẨN (Khoảng 80% sinh viên) ---
        # Sinh viên bình thường nhập học lúc 18 tuổi, hiện tại khoảng 18-22 tuổi
        dob = fake.date_of_birth(minimum_age=18, maximum_age=22)
        enrollment_year = dob.year + 18 
        department_id = random.choice(valid_dept_ids) if valid_dept_ids else None

        # --- INJECT EDGE CASES (Trường hợp ngoại lệ) ---
        rand_val = random.random() # Trả về số ngẫu nhiên từ 0.0 đến 1.0
        
        if rand_val < 0.05:
            # Ngoại lệ 1: Sinh viên năm nhất chưa chọn chuyên ngành (Undeclared Major)
            # Sẽ có khoảng 5% sinh viên có department_id = NULL
            department_id = None 
            
        elif rand_val < 0.08:
            # Ngoại lệ 2: Thần đồng / Sinh viên nhỏ tuổi
            # 3% sinh viên nhập học từ năm 14-16 tuổi
            dob = fake.date_of_birth(minimum_age=14, maximum_age=16)
            enrollment_year = dob.year + random.randint(14, 15)
            
        elif rand_val < 0.12:
            # Ngoại lệ 3: Sinh viên hệ vừa học vừa làm / Lớn tuổi (Adult Learners)
            # 4% sinh viên có độ tuổi từ 30-50, nhập học rất muộn so với tuổi
            dob = fake.date_of_birth(minimum_age=30, maximum_age=50)
            enrollment_year = dob.year + random.randint(20, 30) 
            
        elif rand_val < 0.15:
            # Ngoại lệ 4: Sinh viên "ngâm" quá hạn (Học mãi chưa ra trường)
            # 3% sinh viên có năm nhập học cách đây 8-10 năm (Ví dụ: 2014, 2015)
            dob = fake.date_of_birth(minimum_age=28, maximum_age=32)
            enrollment_year = 2014 + random.randint(0, 2)

        # Chuyển đổi Date object thành chuỗi YYYY-MM-DD để lưu vào SQLite
        dob_str = dob.strftime('%Y-%m-%d')

        fake_data.append((student_id, first_name, last_name, email, dob_str, enrollment_year, department_id))

    # 3. Chèn dữ liệu vào bảng (Sử dụng cursor)
    cursor.executemany('''
        INSERT INTO Student (student_id, first_name, last_name, email, date_of_birth, enrollment_year, department_id)
        VALUES (?, ?, ?, ?, ?, ?, ?)
    ''', fake_data)

    print(f"Đã thêm thành công {n} sinh viên vào bảng Student, bao gồm cả các ngoại lệnh")

#### Hàm tạo dữ liệu cho bảng Course

In [6]:
def feed_course_data(n, cursor):
    # 1. Lấy danh sách department_id để làm Khóa ngoại
    cursor.execute("SELECT department_id FROM Department")
    dept_records = cursor.fetchall()
    valid_dept_ids = [r[0] for r in dept_records] if dept_records else []

    # 2. Lấy course_id lớn nhất hiện tại
    cursor.execute("SELECT MAX(course_id) FROM Course")
    result = cursor.fetchone()
    current_max_id = result[0] if result[0] is not None else 0

    # Dữ liệu nguồn để tạo tên môn học thực tế
    subjects = ["Computer Science", "Mathematics", "Physics", "Economics", "Psychology", 
                "History", "Biology", "Literature", "Chemistry", "Philosophy"]
    levels = ["Introduction to", "Advanced", "Principles of", "Fundamentals of", "Applied"]

    fake_data = []
    generated_codes = set() # Dùng set để theo dõi và đảm bảo course_code là UNIQUE tuyệt đối

    for i in range(1, n + 1):
        course_id = current_max_id + i
        rand_val = random.random()

        # --- DỮ LIỆU CHUẨN (Khoảng 80% môn học) ---
        dept_id = random.choice(valid_dept_ids) if valid_dept_ids else None
        credits = random.randint(2, 4) # Tín chỉ thông thường: 2 đến 4
        
        subj = random.choice(subjects)
        course_name = f"{random.choice(levels)} {subj}"
        
        # Sinh mã môn học (VD: COM101, MAT202). 
        # Đảm bảo mã sinh ra không bị trùng với các mã đã tạo trước đó
        subj_code = subj[:3].upper()
        while True:
            course_code = f"{subj_code}{random.randint(100, 499)}"
            if course_code not in generated_codes:
                generated_codes.add(course_code)
                break

        # --- INJECT EDGE CASES (Trường hợp ngoại lệ) ---
        if rand_val < 0.05:
            # Ngoại lệ 1: Môn học không thuộc Khoa nào (Môn chung toàn trường)
            # Ví dụ: Giáo dục thể chất, Sinh hoạt công dân
            dept_id = None
            course_name = f"University General Orientation {random.randint(1, 5)}"
            course_code = f"UNI{random.randint(100, 199)}"
            while course_code in generated_codes:
                course_code = f"UNI{random.randint(100, 199)}"
            generated_codes.add(course_code)
            
        elif rand_val < 0.10:
            # Ngoại lệ 2: Môn học 0 tín chỉ
            # Các môn bắt buộc nhưng không tính điểm trung bình (GPA)
            credits = 0
            course_name = f"Mandatory Physical Education {random.randint(1, 5)}"
            course_code = f"PE{random.randint(100, 199)}"
            while course_code in generated_codes:
                course_code = f"PE{random.randint(100, 199)}"
            generated_codes.add(course_code)
            
        elif rand_val < 0.15:
            # Ngoại lệ 3: Môn học siêu nặng (Nhiều tín chỉ)
            # Ví dụ: Đồ án tốt nghiệp, Thực tập thực tế (6 đến 12 tín chỉ)
            credits = random.randint(6, 12)
            course_name = f"Graduation Thesis in {subj}"
            course_code = f"{subj_code}499" # Mã 499 thường dành cho đồ án
            while course_code in generated_codes:
                course_code = f"{subj_code}4{random.randint(90, 98)}"
            generated_codes.add(course_code)
            
        elif rand_val < 0.20:
            # Ngoại lệ 4: Tên môn học cực kỳ dài
            # Test xem giao diện UI (nếu có) hoặc các hàm cắt chuỗi SQL có hoạt động tốt không
            credits = 3
            course_name = f"Special Topics: An In-Depth Interdisciplinary Analysis of {subj} and Its Real-World Applications in the Modern 21st Century Era"

        fake_data.append((course_id, course_code, course_name, credits, dept_id))

    # 3. Chèn dữ liệu vào bảng
    cursor.executemany('''
        INSERT INTO Course (course_id, course_code, course_name, credits, department_id)
        VALUES (?, ?, ?, ?, ?)
    ''', fake_data)

    print(f"Đã thêm thành công {n} môn học vào bảng Course (Đã bao gồm các Edge Cases)!")

#### Hàm tạo dữ liệu cho bảng Semester

In [7]:
# Hàm tạo học kì
def feed_semester_data(n, cursor):
    # Lấy semester_id lớn nhất hiện tại
    cursor.execute("SELECT MAX(semester_id) FROM Semester")
    result = cursor.fetchone()
    current_max_id = result[0] if result[0] is not None else 0

    # Cấu hình các học kỳ chuẩn trong một năm (Dựa theo hệ thống Mỹ)
    seasons = ["SPRING", "SUMMER", "FALL"]
    start_months = {"SPRING": 1, "SUMMER": 6, "FALL": 9}
    end_months = {"SPRING": 5, "SUMMER": 8, "FALL": 12}
    
    fake_data = []
    base_year = 2025 # Năm bắt đầu lùi về quá khứ
    
    for i in range(n):
        semester_id = current_max_id + i + 1
        
        # Công thức tính năm và mùa lùi dần để không bao giờ bị trùng lặp
        year = base_year - (i // 3)
        season = seasons[i % 3]
        
        semester_code = f"{season}{year}" # Vd: SPRING2024, FALL2023
        start_date = f"{year}-{start_months[season]:02d}-10" # Thường bắt đầu mùng 10
        end_date = f"{year}-{end_months[season]:02d}-20"     # Thường kết thúc ngày 20
        
        rand_val = random.random()
        
        # --- INJECT EDGE CASES (Trường hợp ngoại lệ) ---
        if i == 0:
            # Ngoại lệ 1: Học kỳ hiện tại đang diễn ra (Chưa có ngày kết thúc)
            # Cột end_date sẽ là NULL
            end_date = None
            
        elif i == 1:
            # Ngoại lệ 2: Học kỳ phụ cực ngắn (Winter Mini-Mester)
            # Vắt ngang giữa 2 năm (Bắt đầu cuối tháng 12 năm nay, kết thúc giữa tháng 1 năm sau)
            semester_code = f"WINTER{year}"
            start_date = f"{year}-12-26"
            end_date = f"{year + 1}-01-14"

        fake_data.append((semester_id, semester_code, start_date, end_date))

    # Chèn dữ liệu vào bảng
    cursor.executemany('''
        INSERT INTO Semester (semester_id, semester_code, start_date, end_date)
        VALUES (?, ?, ?, ?)
    ''', fake_data)

    print(f"Đã thêm thành công {n} học kỳ (Đã bao gồm các Edge Cases)!")

#### Hàm tạo dữ liệu cho bảng Classroom

In [8]:
# Hàm tạo lớp học
def feed_classroom_data(n, cursor):
    """
    Hàm sinh dữ liệu CƠ BẢN cho bảng Classroom.
    (Giữ n = 15 cho dễ thao tác)
    """
    # Lấy ID hiện tại
    cursor.execute("SELECT MAX(classroom_id) FROM Classroom")
    max_id = cursor.fetchone()[0] or 0

    fake_data = []
    buildings = ["Building A", "Building B", "Building C"]

    for i in range(1, n + 1):
        classroom_id = max_id + i
        building = random.choice(buildings)
        room_number = f"{random.randint(1, 9)}0{random.randint(1, 9)}" # Vd: 101, 305
        
        # Sức chứa: Hoặc 30, hoặc 50, hoặc 100, thỉnh thoảng có giá trị NULL
        capacity = random.choice([30, 50, 100, None]) 

        fake_data.append((classroom_id, building, room_number, capacity))

    cursor.executemany('''
        INSERT INTO Classroom (classroom_id, building, room_number, capacity)
        VALUES (?, ?, ?, ?)
    ''', fake_data)
    print(f"Đã thêm {n} dòng vào Classroom.")

#### Hàm tạo dữ liệu cho bảng Class Section

In [9]:
def feed_class_section_data(n, cursor):
    """
    Hàm sinh dữ liệu CƠ BẢN cho bảng Class_Section.
    (Giữ n = 100 cho dễ thao tác)
    """
    # 1. Lấy tất cả Khóa ngoại cần thiết
    cursor.execute("SELECT course_id FROM Course")
    course_ids = [row[0] for row in cursor.fetchall()]

    cursor.execute("SELECT professor_id FROM Professor")
    prof_ids = [row[0] for row in cursor.fetchall()]

    cursor.execute("SELECT semester_id FROM Semester")
    sem_ids = [row[0] for row in cursor.fetchall()]

    cursor.execute("SELECT classroom_id FROM Classroom")
    room_ids = [row[0] for row in cursor.fetchall()]

    # Kiểm tra an toàn
    if not all([course_ids, prof_ids, sem_ids, room_ids]):
        print("Cảnh báo: Thiếu dữ liệu ở một trong các bảng cha. Không thể tạo Class_Section.")
        return

    # 2. Lấy ID hiện tại
    cursor.execute("SELECT MAX(section_id) FROM Class_Section")
    max_id = cursor.fetchone()[0] or 0

    fake_data = []

    for i in range(1, n + 1):
        section_id = max_id + i
        course_id = random.choice(course_ids)
        semester_id = random.choice(sem_ids)
        
        # Thỉnh thoảng lớp chưa có giảng viên (NULL)
        professor_id = random.choice(prof_ids + [None])
        
        # Thỉnh thoảng lớp chưa xếp phòng (NULL)
        classroom_id = random.choice(room_ids + [None]) 

        fake_data.append((section_id, course_id, professor_id, semester_id, classroom_id))

    cursor.executemany('''
        INSERT INTO Class_Section (section_id, course_id, professor_id, semester_id, classroom_id)
        VALUES (?, ?, ?, ?, ?)
    ''', fake_data)
    print(f"Đã thêm {n} dòng vào Class_Section.")

#### Hàm tạo dữ liệu cho bảng Enrollment

In [10]:
def feed_enrollment_data(n, cursor):
    """
    Hàm sinh dữ liệu CƠ BẢN cho bảng Enrollment (Đăng ký học/Bảng điểm).
    (Khuyên dùng: n = 2500)
    """
    cursor.execute("SELECT student_id FROM Student")
    student_ids = [row[0] for row in cursor.fetchall()]
    
    cursor.execute("SELECT section_id FROM Class_Section")
    section_ids = [row[0] for row in cursor.fetchall()]

    if not student_ids or not section_ids:
        print("Lỗi: Thiếu dữ liệu từ Student hoặc Class_Section.")
        return

    cursor.execute("SELECT MAX(enrollment_id) FROM Enrollment")
    max_id = cursor.fetchone()[0] or 0
    fake_data = []

    for i in range(1, n + 1):
        enrollment_id = max_id + i
        student_id = random.choice(student_ids)
        section_id = random.choice(section_ids)
        
        # Điểm số: Random từ 0.00 đến 10.00 (Làm tròn 2 chữ số thập phân)
        # Thỉnh thoảng trả về NULL (Sinh viên chưa có điểm)
        final_grade = round(random.uniform(0.0, 10.0), 2) if random.random() > 0.15 else None

        fake_data.append((enrollment_id, student_id, section_id, final_grade))

    cursor.executemany('''
        INSERT INTO Enrollment (enrollment_id, student_id, section_id, final_grade)
        VALUES (?, ?, ?, ?)
    ''', fake_data)
    print(f"Đã thêm {n} dòng vào Enrollment.")

#### Hàm tạo dữ liệu cho bảng Scholarship

In [11]:
def feed_scholarship_data(n, cursor):
    """
    Hàm sinh dữ liệu CƠ BẢN cho bảng Scholarship (Danh mục học bổng).
    (Khuyên dùng: n = 10)
    """
    cursor.execute("SELECT MAX(scholarship_id) FROM Scholarship")
    max_id = cursor.fetchone()[0] or 0
    fake_data = []
    
    sponsors = ["Tech Foundation", "Alumni Association", "City Council", "Global Edu"]

    for i in range(1, n + 1):
        scholarship_id = max_id + i
        scholarship_name = f"Excellence Award Type {random.randint(10, 99)}"
        # Số tiền: Từ 1,000.00 đến 10,000.00
        amount = round(random.uniform(1000.0, 10000.0), 2)
        sponsor_name = random.choice(sponsors)

        fake_data.append((scholarship_id, scholarship_name, amount, sponsor_name))

    cursor.executemany('''
        INSERT INTO Scholarship (scholarship_id, scholarship_name, amount, sponsor_name)
        VALUES (?, ?, ?, ?)
    ''', fake_data)
    print(f"Đã thêm {n} dòng vào Scholarship.")

#### Hàm tạo dữ liệu cho bảng trung gian Student và Scholarship

In [12]:
def feed_student_scholarship_data(n, cursor):
    """
    Hàm sinh dữ liệu CƠ BẢN cho bảng Student_Scholarship.
    (Khuyên dùng: n = 50)
    """
    cursor.execute("SELECT student_id FROM Student")
    student_ids = [row[0] for row in cursor.fetchall()]
    
    cursor.execute("SELECT scholarship_id FROM Scholarship")
    scholarship_ids = [row[0] for row in cursor.fetchall()]

    if not student_ids or not scholarship_ids:
        print("Lỗi: Thiếu dữ liệu từ Student hoặc Scholarship.")
        return

    cursor.execute("SELECT MAX(award_id) FROM Student_Scholarship")
    max_id = cursor.fetchone()[0] or 0
    fake_data = []

    for i in range(1, n + 1):
        award_id = max_id + i
        student_id = random.choice(student_ids)
        scholarship_id = random.choice(scholarship_ids)
        
        # Ngày cấp: Định dạng chuẩn YYYY-MM-DD
        year = random.randint(2020, 2024)
        month = random.randint(1, 12)
        day = random.randint(1, 28)
        award_date = f"{year}-{month:02d}-{day:02d}"

        fake_data.append((award_id, student_id, scholarship_id, award_date))

    cursor.executemany('''
        INSERT INTO Student_Scholarship (award_id, student_id, scholarship_id, award_date)
        VALUES (?, ?, ?, ?)
    ''', fake_data)
    print(f"Đã thêm {n} dòng vào Student_Scholarship.")

### Thực thi hàm tạo dữ liệu cho database

In [13]:
feed_department_data(5, cursor)
feed_professor_data(25, cursor)
feed_student_data(500, cursor)
feed_course_data(40, cursor)
feed_semester_data(6, cursor)
feed_classroom_data(10, cursor)
feed_class_section_data(100,cursor)
feed_enrollment_data(2500, cursor)
feed_scholarship_data(10, cursor)
feed_student_scholarship_data(50, cursor)

Current max id: 0
Thêm 5 dòng dữ liệu vào bảng Department thành công !
Thêm 25 dòng dữ liệu vào bảng Professor thành công !
Đã thêm thành công 500 sinh viên vào bảng Student, bao gồm cả các ngoại lệnh
Đã thêm thành công 40 môn học vào bảng Course (Đã bao gồm các Edge Cases)!
Đã thêm thành công 6 học kỳ (Đã bao gồm các Edge Cases)!
Đã thêm 10 dòng vào Classroom.
Đã thêm 100 dòng vào Class_Section.
Đã thêm 2500 dòng vào Enrollment.
Đã thêm 10 dòng vào Scholarship.
Đã thêm 50 dòng vào Student_Scholarship.


In [14]:
conn.commit()
conn.close()